In [1]:
import os
import re
import math
import random
from collections import Counter
from tqdm import tqdm

import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
window_size = 100
batch_size = 128
embedding_dim = 128
hidden_size = 256
num_layers = 2
lr = 0.001
epochs = 8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
min_freq = 1
print("Device:", device)

Device: cpu


In [3]:
def download_text(url):
    try:
        print("Downloading text...")
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        return r.text
    except Exception as e:
        print("Download failed:", e)
        return None

gutenberg_txt_url = "https://www.gutenberg.org/cache/epub/84/pg84.txt"
raw = download_text(gutenberg_txt_url)
if raw is None:
    print("No internet — using fallback sample text.")
    raw = ("In the quiet village the sun rose slowly. " * 200)
print("Loaded chars:", len(raw))

Loaded chars: 446583


In [4]:
def clean_and_tokenize(text):
    txt = text.replace("\r\n", "\n")
    txt = re.sub(r"\s+", " ", txt).strip()
    txt = txt.lower()
    tokens = re.findall(r"\w+|[^\s\w]", txt, re.UNICODE)
    return tokens

tokens = clean_and_tokenize(raw)
print("Total tokens:", len(tokens))
if len(tokens) < window_size + 10:
    repeats = math.ceil((window_size + 10) / max(1, len(tokens)))
    tokens = tokens * repeats
    print("Text repeated", repeats, "times -> tokens:", len(tokens))

Total tokens: 89677


In [5]:
counter = Counter(tokens)
vocab_tokens = [w for w, c in counter.items() if c >= min_freq]
vocab_tokens.sort(key=lambda w: (-counter[w], w))

PAD = "<PAD>"
UNK = "<UNK>"
START = "<START>"

itos = [PAD, UNK, START] + vocab_tokens
stoi = {w: i for i, w in enumerate(itos)}
vocab_size = len(itos)
print("Vocab size:", vocab_size)

def tok2idx(tok):
    return stoi.get(tok, stoi[UNK])

seq_len = window_size - 1
inputs = []
targets = []
for i in range(0, len(tokens) - seq_len):
    inp = tokens[i : i + seq_len]
    targ = tokens[i + seq_len]
    inputs.append([tok2idx(t) for t in inp])
    targets.append(tok2idx(targ))

print("Number of sequences:", len(inputs))

Vocab size: 7366
Number of sequences: 89578


In [6]:
class TextSeqDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs = inputs
        self.targets = targets
    def __len__(self):
        return len(self.inputs)
    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx], dtype=torch.long), torch.tensor(self.targets[idx], dtype=torch.long)

dataset = TextSeqDataset(inputs, targets)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

In [7]:
class LSTMGenerator(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
    def forward(self, x, hidden=None):
        # x: (batch, seq_len)
        emb = self.embedding(x)
        out, hidden = self.lstm(emb, hidden)
        out = out[:, -1, :]
        logits = self.fc(out)
        return logits, hidden

model = LSTMGenerator(vocab_size, embedding_dim, hidden_size, num_layers=num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
print(model)

LSTMGenerator(
  (embedding): Embedding(7366, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=256, out_features=7366, bias=True)
)


In [8]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for xb, yb in dataloader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits, _ = model(xb)   # we don't carry hidden across batches here
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(dataloader.dataset)

for epoch in range(1, epochs+1):
    loss = train_epoch(model, dataloader, optimizer, criterion, device)
    print(f"Epoch {epoch}/{epochs} — loss: {loss:.4f}")

Epoch 1/8 — loss: 6.2087
Epoch 2/8 — loss: 5.5546
Epoch 3/8 — loss: 5.2361
Epoch 4/8 — loss: 4.9851
Epoch 5/8 — loss: 4.7438
Epoch 6/8 — loss: 4.5112
Epoch 7/8 — loss: 4.2835
Epoch 8/8 — loss: 4.0599


In [10]:
import torch

def generate_text(model, seed_text, max_words=120, temperature=1.0):
    model.eval()
    words = clean_and_tokenize(seed_text)
    while len(words) < seq_len:
        words = [START] + words
    generated = words[:]
    hidden = None
    for _ in range(max_words):
        input_seq = generated[-seq_len:]
        idxs = torch.tensor([tok2idx(w) for w in input_seq], dtype=torch.long).unsqueeze(0).to(device)  # (1, seq_len)
        with torch.no_grad():
            logits, hidden = model(idxs, hidden)   # hidden is tuple (h_n, c_n)
            logits = logits.squeeze(0) / (temperature if temperature>0 else 1.0)
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()
            next_word = itos[next_idx] if next_idx < len(itos) else UNK
            generated.append(next_word)

    out = []
    for token in generated:
        if re.match(r"^\w+$", token):
            if out:
                out.append(" ")
            out.append(token)
        else:
            out.append(token)
    return "".join(out)

seed = "the monster"
print(generate_text(model, seed, max_words=120, temperature=1.0))

<START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START><START> the monster uncontrollably protected to support, before me. it was near his own as that it was not to that it was complete on for the turk that could not now most not exchanged their subjects. even i know not remember that it were only for a more project gutenberg work, without fellow work. if if if chapter 19 can rece

In [11]:
torch.save({
    "model_state_dict": model.state_dict(),
    "stoi": stoi,
    "itos": itos,
    "config": {
        "window_size": window_size,
        "embedding_dim": embedding_dim,
        "hidden_size": hidden_size,
        "num_layers": num_layers
    }
}, "lstm_text_gen.pth")
print("Saved model -> lstm_text_gen.pth")

Saved model -> lstm_text_gen.pth
